# AROME Daily Fetch — France Metropolitan Grid

Self-healing archive: queries the API for all available coverages,
compares against the local cache, downloads only what's missing.

Two modes:
- **Static** (daily cron): 00Z run, first 24h → analysis-quality daily archive
- **Dynamic** (every 3h): latest window → rolling near-real-time

Run this notebook (or the `sync()` function) any time — it catches up
automatically. If you missed 3 days, next run fetches all 3.

```
data/arome_daily/
  static/arome_daily_2026-07-15.nc    ← full 24h, analysis quality
  dynamic/arome_daily_2026-07-15.nc   ← assembled from 3h windows
```

In [1]:
import logging
import os
import re
import time
from collections import defaultdict
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import xarray as xr

REPO_ROOT = os.path.dirname(os.getcwd())
os.chdir(REPO_ROOT)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
logger = logging.getLogger("arome")

from irrigator.utils.auth_meteofrance import meteo_headers

In [2]:
# === Configuration ===

FRANCE_BBOX = {"north": 51.5, "south": 41.0, "west": -6.0, "east": 10.0}

RAW_DIR = Path("data/arome_raw")
DAILY_DIR = Path("data/arome_daily")
for d in [RAW_DIR / "static", RAW_DIR / "dynamic", DAILY_DIR / "static", DAILY_DIR / "dynamic"]:
    d.mkdir(parents=True, exist_ok=True)

DELETE_RAW = False
API_PAUSE = 0.5

BASE_URL = "https://public-api.meteofrance.fr/public/arome/1.0"
WCS_RESOURCE = "wcs/MF-NWP-HIGHRES-AROME-001-FRANCE-WCS"

---
## Step 0: Discover available coverages from the API

In [3]:
def get_available_coverages() -> list[str]:
    """Query GetCapabilities and return all available coverage IDs."""
    resp = requests.get(
        f"{BASE_URL}/{WCS_RESOURCE}/GetCapabilities",
        params={"service": "WCS", "version": "2.0.1", "language": "eng"},
        headers=meteo_headers(),
        timeout=60,
    )
    resp.raise_for_status()
    
    coverages = []
    for line in resp.text.splitlines():
        if "CoverageId" in line:
            cid = line.strip().replace("<wcs:CoverageId>", "").replace("</wcs:CoverageId>", "")
            if cid:
                coverages.append(cid)
    return coverages


def parse_coverage_id(cid: str) -> dict | None:
    """Extract variable, run datetime, and accumulation from a coverage ID.
    
    Examples:
      TOTAL_PRECIPITATION__GROUND_OR_WATER_SURFACE___2026-06-10T00.00.00Z_PT1H
      TEMPERATURE__GROUND_OR_WATER_SURFACE___2026-06-10T00.00.00Z
    """
    # Split on the triple underscore that separates variable from datetime
    parts = cid.split("___")
    if len(parts) != 2:
        return None
    
    variable = parts[0]
    datetime_part = parts[1]  # e.g., "2026-06-10T00.00.00Z_PT1H" or "2026-06-10T00.00.00Z"
    
    # Split accumulation if present
    accum = None
    # Match: date part then optional _P... accumulation
    m = re.match(r"(\d{4}-\d{2}-\d{2}T\d{2}\.\d{2}\.\d{2}Z)(?:_(P.+))?", datetime_part)
    if not m:
        return None
    
    dt_str = m.group(1)
    accum = m.group(2)  # None if no accumulation
    
    dt = datetime.strptime(dt_str, "%Y-%m-%dT%H.%M.%SZ").replace(tzinfo=timezone.utc)
    
    return {
        "coverage_id": cid,
        "variable": variable,
        "run_datetime": dt,
        "run_date": dt.date(),
        "run_hour": dt.hour,
        "accumulation": accum,
    }

In [4]:
# Fetch and parse
raw_coverages = get_available_coverages()
parsed = [parse_coverage_id(c) for c in raw_coverages]
parsed = [p for p in parsed if p is not None]

# Index by variable
by_variable = defaultdict(list)
for p in parsed:
    by_variable[p["variable"]].append(p)

print(f"Total coverages: {len(parsed)}")
print(f"Unique variables: {len(by_variable)}")
print()
for var, items in sorted(by_variable.items()):
    dates = sorted(set(p["run_date"] for p in items))
    accums = sorted(set(p["accumulation"] or "instant" for p in items))
    print(f"  {var[:60]:60s} dates={dates[0]}→{dates[-1]}  accum={accums}")

Total coverages: 6270
Unique variables: 46

  AVERAGE_LIGHTNING_STRIKE_DENSITY_OVER_3HOURS__GROUND_OR_WATE dates=2026-06-09→2026-06-13  accum=['instant']
  BRIGHTNESS_TEMPERATURE__GROUND_OR_WATER_SURFACE              dates=2026-06-09→2026-06-13  accum=['instant']
  CONVECTIVE_AVAILABLE_POTENTIAL_ENERGY__GROUND_OR_WATER_SURFA dates=2026-06-09→2026-06-13  accum=['instant']
  CONVECTIVE_INHIBITION__GROUND_OR_WATER_SURFACE               dates=2026-06-09→2026-06-13  accum=['instant']
  DEW_POINT_TEMPERATURE__SPECIFIC_HEIGHT_LEVEL_ABOVE_GROUND    dates=2026-06-09→2026-06-13  accum=['instant']
  DOWNWARD_DIRECT_SHORT_WAVE_RADIATION_FLUX__GROUND_OR_WATER_S dates=2026-06-09→2026-06-13  accum=['P1D', 'P2D', 'PT12H', 'PT18H', 'PT1H', 'PT3H', 'PT6H', 'PT9H']
  DOWNWARD_LONG_WAVE_RADIATION_FLUX__GROUND_OR_WATER_SURFACE   dates=2026-06-09→2026-06-13  accum=['P1D', 'P2D', 'PT12H', 'PT18H', 'PT1H', 'PT3H', 'PT6H', 'PT9H']
  DOWNWARD_SHORT_WAVE_RADIATION_FLUX__GROUND_OR_WATER_SURFACE  dates=2026-06-09→

---
## Variable mapping and fetch logic

In [5]:
# Map our short names to API coverage base names
VARS_INSTANTANEOUS = {
    "temp_2m": {
        "coverage": "TEMPERATURE__GROUND_OR_WATER_SURFACE",
        "height": None,
        "static_hours": list(range(0, 24)),   # all 24 for Tmin/Tmax
        "dynamic_hours": [0, 1, 2],                  # single snapshot per window
    },
    "wind_10m": {
        "coverage": "WIND_SPEED__SPECIFIC_HEIGHT_LEVEL_ABOVE_GROUND",
        "height": "10",
        "static_hours": list(range(0, 24)), # 3-hourly for daily mean
        "dynamic_hours": [0, 1, 2],
    },
    "dewpoint_2m": {
        "coverage": "DEW_POINT_TEMPERATURE__SPECIFIC_HEIGHT_LEVEL_ABOVE_GROUND",
        "height": "2",
        "static_hours": list(range(0, 24)),
        "dynamic_hours": [0, 1, 2],
    },
}

VARS_ACCUMULATED = {
    "precip": {
        "coverage": "TOTAL_PRECIPITATION__GROUND_OR_WATER_SURFACE",
        "static_accum": "P1D",     # 1 call, full day total
        "dynamic_accum": "PT3H",   # 1 call per 3h window
    },
    "solar_rad": {
        "coverage": "DOWNWARD_SHORT_WAVE_RADIATION_FLUX__GROUND_OR_WATER_SURFACE",
        "static_accum": "P1D",
        "dynamic_accum": "PT3H",
    },
}


def find_available_run_dates(
    parsed_coverages: list[dict],
    run_hour: int = 0,
) -> list[date]:
    """Extract unique run dates for a specific run hour from capabilities."""
    dates = set()
    for p in parsed_coverages:
        if p["run_hour"] == run_hour:
            dates.add(p["run_date"])
    return sorted(dates)


available_00z_dates = find_available_run_dates(parsed, run_hour=0)
print(f"Available 00Z dates: {available_00z_dates[0]} → {available_00z_dates[-1]} ({len(available_00z_dates)} days)")

Available 00Z dates: 2026-06-09 → 2026-06-13 (5 days)


---
## Core fetch: single timestep

In [6]:
def fetch_single_timestep(
    coverage_id: str,
    valid_time_utc: datetime,
    out_path: Path,
    bbox: dict = FRANCE_BBOX,
    height: str | None = None,
    overwrite: bool = False,
) -> Path:
    """Fetch one AROME grid for one timestep."""
    if out_path.exists() and not overwrite:
        return out_path

    session = requests.Session()
    session.headers.update(meteo_headers())

    time_str = valid_time_utc.strftime("%Y-%m-%dT%H:%M:%SZ")
    params = {
        "service": "WCS",
        "version": "2.0.1",
        "request": "GetCoverage",
        "coverageId": coverage_id,
        "subset": [
            f"time({time_str})",
            f"lat({bbox['south']},{bbox['north']})",
            f"long({bbox['west']},{bbox['east']})",
        ],
        "format": "application/wmo-grib",
    }
    if height:
        params["subset"].append(f"height({height})")

    resp = session.get(f"{BASE_URL}/{WCS_RESOURCE}/GetCoverage", params=params, timeout=120)
    if not resp.ok:
        logger.error("FAIL %s at %s: %d", coverage_id[:50], time_str, resp.status_code)
        resp.raise_for_status()

    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_bytes(resp.content)
    return out_path

---
## Static fetch for one date

In [7]:
def format_cid(coverage_base: str, run_date: date, accum: str | None = None) -> str:
    dt_str = f"{run_date.isoformat()}T00.00.00Z"
    cid = f"{coverage_base}___{dt_str}"
    if accum:
        cid += f"_{accum}"
    return cid


def fetch_static_day(target_date: date, overwrite: bool = False) -> Path:
    """Fetch all variables for one day from the 00Z run (static mode).
    
    ~42 API calls: precip(1) + rad(1) + temp(24) + wind(8) + dew(8)
    """
    day_dir = RAW_DIR / "static" / target_date.isoformat()
    run_start = datetime(target_date.year, target_date.month, target_date.day, tzinfo=timezone.utc)
    calls = 0

    # Accumulated: P1D, valid at step 24h
    for var_name, cfg in VARS_ACCUMULATED.items():
        cid = format_cid(cfg["coverage"], target_date, cfg["static_accum"])
        valid_t = run_start + timedelta(hours=24)
        out = day_dir / f"{var_name}_P1D.grib2"
        fetch_single_timestep(cid, valid_t, out, height=None, overwrite=overwrite)
        calls += 1
        time.sleep(API_PAUSE)

    # Instantaneous: per hour or per 3h
    for var_name, cfg in VARS_INSTANTANEOUS.items():
        cid = format_cid(cfg["coverage"], target_date)
        for h in cfg["static_hours"]:
            valid_t = run_start + timedelta(hours=h)
            out = day_dir / f"{var_name}_{h:02d}.grib2"
            fetch_single_timestep(cid, valid_t, out, height=cfg["height"], overwrite=overwrite)
            calls += 1
            time.sleep(API_PAUSE)

    logger.info("[static] %s: %d calls", target_date, calls)
    return day_dir

---
## Dynamic fetch for one 3h window

In [8]:
def fetch_dynamic_window(
    run_date: date,
    run_hour: int,
    forecast_hour: int,
    overwrite: bool = False,
) -> Path:
    """Fetch one 3h window (5 API calls)."""
    run_start = datetime(run_date.year, run_date.month, run_date.day,
                         run_hour, tzinfo=timezone.utc)
    valid_t = run_start + timedelta(hours=forecast_hour)
    tag = valid_t.strftime("%Y-%m-%dT%H")
    window_dir = RAW_DIR / "dynamic" / tag

    for var_name, cfg in VARS_ACCUMULATED.items():
        cid = format_cid(cfg["coverage"], run_date, cfg["dynamic_accum"])
        out = window_dir / f"{var_name}_PT3H.grib2"
        fetch_single_timestep(cid, valid_t + timedelta(hours=3), out, overwrite=overwrite)
        time.sleep(API_PAUSE)

    for var_name, cfg in VARS_INSTANTANEOUS.items():
        cid = format_cid(cfg["coverage"], run_date)
        for h in cfg["dynamic_hours"]:
            valid_t_sub = valid_t + timedelta(hours=h)
            out = window_dir / f"{var_name}_{h:02d}.grib2"
            fetch_single_timestep(cid, valid_t_sub, out, height=cfg["height"], overwrite=overwrite)
            time.sleep(API_PAUSE)

    logger.info("[dynamic] %s: done", tag)
    return window_dir

---
## Build daily NetCDF from raw GRIBs

In [11]:
from irrigator.ingestion.meteofrance_client import open_forecast


def _open_grib_scalar(path: Path,*, is_surface : bool = False) -> xr.DataArray:
    ds = open_forecast(path)
    ds = ds.drop_vars(["surface"]) if is_surface else ds.drop_vars(["heightAboveGround"])
    #ds = ds.rename({"surface": "heightAboveGround"}) if is_surface else ds
    return ds[list(ds.data_vars)[0]]


def build_daily(target_date: date, mode: str = "static") -> Path:
    """Build daily NetCDF from raw GRIBs."""
    out_path = DAILY_DIR / mode / f"arome_daily_{target_date.isoformat()}.nc"
    if mode == "static":
        return _build_static(target_date, out_path)
    else:
        return _build_dynamic(target_date, out_path)


def _build_static(target_date: date, out_path: Path) -> Path:
    day_dir = RAW_DIR / "static" / target_date.isoformat()
    if not day_dir.exists():
        raise FileNotFoundError(f"No static raw for {target_date}")

    daily = {}

    print("0")

    # Accumulated: single P1D file = daily total
    pf = day_dir / "precip_P1D.grib2"
    if pf.exists():
        daily["precip_mm"] = _open_grib_scalar(pf, is_surface=True)  # kg/m² = mm
    
    rf = day_dir / "solar_rad_P1D.grib2"
    if rf.exists():
        daily["rs_mj"] = _open_grib_scalar(rf, is_surface=True) / 1e6  # J → MJ

    # Temperature: hourly → min/max/mean
    temp_files = sorted(day_dir.glob("temp_2m_*.grib2"))
    if temp_files:
        temps = xr.concat([_open_grib_scalar(f, is_surface=True) for f in temp_files], dim="step")
        daily["t_min"] = temps.min(dim="step") - 273.15
        daily["t_max"] = temps.max(dim="step") - 273.15
        daily["t_mean"] = temps.mean(dim="step") - 273.15

    # Wind: 3-hourly → mean
    wind_files = sorted(day_dir.glob("wind_10m_*.grib2"))
    if wind_files:
        daily["wind_speed_10m"] = xr.concat(
            [_open_grib_scalar(f) for f in wind_files], dim="step"
        ).mean(dim="step")

    # Dewpoint: 3-hourly → mean, K → °C
    dew_files = sorted(day_dir.glob("dewpoint_2m_*.grib2"))
    if dew_files:
        daily["dewpoint"] = xr.concat(
            [_open_grib_scalar(f) for f in dew_files], dim="step"
        ).mean(dim="step") - 273.

    ds = xr.Dataset(
        {k: v.expand_dims(valid_time=[np.datetime64(target_date)]) for k, v in daily.items()}
    )
    ds.attrs["source"] = f"AROME 00Z {target_date} (static)"
    encoding = {v: {"zlib": True, "complevel": 4} for v in ds.data_vars}
    ds.to_netcdf(out_path, encoding=encoding)
    logger.info("[static] %s → %.1f MB", out_path.name, out_path.stat().st_size / 1e6)
    
    return out_path


def _build_dynamic(target_date: date, out_path: Path) -> Path:
    pattern = f"{target_date.isoformat()}T*"
    window_dirs = sorted((RAW_DIR / "dynamic").glob(pattern))
    if not window_dirs:
        raise FileNotFoundError(f"No dynamic windows for {target_date}")

    daily = {}
    # Accumulated: sum 3h windows
    precip_parts = [
        _open_grib_scalar(wd / "precip_PT3H.grib2", is_surface=True)
        for wd in window_dirs
        if (wd / "precip_PT3H.grib2").exists()
    ]
    if precip_parts:
        daily["precip_mm"] = sum(precip_parts)
    
    rad_parts = [
        _open_grib_scalar(wd / "solar_rad_PT3H.grib2", is_surface=True)
        for wd in window_dirs
        if (wd / "solar_rad_PT3H.grib2").exists()
    ]
    if rad_parts:
        daily["rs_mj"] = sum(rad_parts) / 1e6
    # Instantaneous: aggregate across windows
    def _collect(name):
        return [
            _open_grib_scalar(wd / f"{name}_*.grib2", is_surface=True)
            for wd in window_dirs
            if (wd / f"{name}.grib2").exists()
        ]

    tv = _collect("temp_2m")
    if tv:
        t = xr.concat(tv, dim="step")
        daily["t_min"] = t.min(dim="step") - 273.15
        daily["t_max"] = t.max(dim="step") - 273.15
        daily["t_mean"] = t.mean(dim="step") - 273.15
    wv = _collect("wind_10m")
    if wv:
        daily["wind_speed_10m"] = xr.concat(wv, dim="step").mean(dim="step")

    dv = _collect("dewpoint_2m")
    if dv:
        daily["dewpoint"] = xr.concat(dv, dim="step").mean(dim="step") - 273.15

    n_win = len(window_dirs)

    ds = xr.Dataset(
        {k: v.expand_dims(valid_time=[np.datetime64(target_date)]) for k, v in daily.items()}
    )
    ds.attrs["source"] = f"AROME dynamic {target_date} ({n_win} windows)"
    ds.attrs["n_windows"] = n_win
    ds.attrs["complete"] = int(n_win >= 8)
    encoding = {v: {"zlib": True, "complevel": 4} for v in ds.data_vars}
    ds.to_netcdf(out_path, encoding=encoding)
    logger.info("[dynamic] %s → %d windows, %.1f MB", out_path.name, n_win, out_path.stat().st_size / 1e6)
    return out_path

---
## Sync: discover available → compare cache → fetch missing

In [10]:
def sync_static(overwrite: bool = False, delete_raw : bool = DELETE_RAW) -> dict:
    """Discover all available 00Z dates, fetch and build any missing.
    
    Self-healing: run any time, catches up automatically.
    Returns {date: status} for each available date.
    """
    logger.info("Syncing static archive...")
    all_coverages = get_available_coverages()
    parsed_all = [parse_coverage_id(c) for c in all_coverages]
    parsed_all = [p for p in parsed_all if p is not None]
    available_dates = find_available_run_dates(parsed_all, run_hour=0)
    
    results = {}
    
    for d in available_dates:
        cache_path = DAILY_DIR / "static" / f"arome_daily_{d.isoformat()}.nc"
        
        if cache_path.exists() and not overwrite:
            results[d] = "cached"
            continue
        
        try:
            fetch_static_day(d, overwrite=overwrite)
            build_daily(d, mode="static")
            
            # Clean raw
            if delete_raw:
                import shutil
                raw_dir = RAW_DIR / "static" / d.isoformat()
                if raw_dir.exists():
                    shutil.rmtree(raw_dir)
            
            results[d] = "fetched"
        except Exception as e:
            logger.error("[static] %s FAILED: %s", d, e)
            results[d] = f"error: {e}"
    
    n_cached = sum(1 for v in results.values() if v == "cached")
    n_fetched = sum(1 for v in results.values() if v == "fetched")
    n_errors = sum(1 for v in results.values() if v.startswith("error"))
    
    logger.info(
        "Sync complete: %d available, %d cached, %d fetched, %d errors",
        len(results), n_cached, n_fetched, n_errors,
    )
    return results


def sync_dynamic(overwrite: bool = False) -> dict:
    """Discover all available run times, fetch missing 3h windows."""
    logger.info("Syncing dynamic archive...")
    all_coverages = get_available_coverages()
    parsed_all = [parse_coverage_id(c) for c in all_coverages]
    parsed_all = [p for p in parsed_all if p is not None]
    
    # Group by run date + run hour
    runs = set()
    for p in parsed_all:
        runs.add((p["run_date"], p["run_hour"]))
    
    results = {}
    
    for run_date, run_hour in sorted(runs):
        # For each run, fetch short-range windows (0, 3, 6h ahead)
        run_start = datetime(run_date.year, run_date.month, run_date.day,
                                run_hour, tzinfo=timezone.utc)
        valid_t = run_start #+ timedelta(hours=fh)
        tag = valid_t.strftime("%Y-%m-%dT%H")
        window_dir = RAW_DIR / "dynamic" / tag
        
        # Check if we already have all files for this window
        expected_files = len(VARS_ACCUMULATED) + len(VARS_INSTANTANEOUS)
        existing = len(list(window_dir.glob("*.grib2"))) if window_dir.exists() else 0
        
        if existing >= expected_files and not overwrite: # temp fix with -1 because of solar_rad
            results[tag] = "cached"
            continue
        
        try:
            fetch_dynamic_window(run_date, run_hour, 0, overwrite=overwrite) # replaced fh by 0
            results[tag] = "fetched"
        except Exception as e:
            results[tag] = f"error: {e}"
    
    # Build daily archives from all windows
    available_days = set()
    for tag in results:
        d = date.fromisoformat(tag[:10])
        available_days.add(d)
    
    for d in sorted(available_days):
        try:
            build_daily(d, mode="dynamic")
        except Exception as e:
            logger.error("[dynamic] build %s failed: %s", d, e)
    
    return results

In [12]:
# === RUN STATIC SYNC ===
# Discovers all available 00Z dates, downloads missing ones, builds daily archives.

results = sync_static(overwrite=False)

for d, status in sorted(results.items()):
    marker = "✓" if status == "cached" else ("↓" if status == "fetched" else "✗")
    print(f"  {marker} {d}: {status}")

2026-06-13 21:34:20,972 Syncing static archive...
2026-06-13 21:36:09,294 [static] 2026-06-13: 74 calls


0


2026-06-13 21:36:14,872 [static] arome_daily_2026-06-13.nc → 19.5 MB
2026-06-13 21:36:14,873 Sync complete: 5 available, 4 cached, 1 fetched, 0 errors


  ✓ 2026-06-09: cached
  ✓ 2026-06-10: cached
  ✓ 2026-06-11: cached
  ✓ 2026-06-12: cached
  ↓ 2026-06-13: fetched


In [13]:
# === RUN DYNAMIC SYNC (optional, for near-real-time) ===
results_dyn = sync_dynamic(overwrite=False)

# ISSUE WITH THE FH BEING 0,3,6 : I THINK IT'S GOOD WHEN NO NEW IS AVAILABLE, BUT THE ISSUE COMES BECAUSE ITS OVER CONSUMING API CALLS FOR OLDER DATA

2026-06-13 21:36:14,882 Syncing dynamic archive...
2026-06-13 21:36:32,624 [dynamic] 2026-06-12T18: done
2026-06-13 21:36:48,041 [dynamic] 2026-06-12T21: done
2026-06-13 21:37:02,750 [dynamic] 2026-06-13T00: done
2026-06-13 21:37:17,445 [dynamic] 2026-06-13T03: done
2026-06-13 21:37:32,522 [dynamic] 2026-06-13T06: done
2026-06-13 21:37:47,360 [dynamic] 2026-06-13T09: done
2026-06-13 21:38:02,095 [dynamic] 2026-06-13T12: done
2026-06-13 21:38:16,844 [dynamic] 2026-06-13T15: done
2026-06-13 21:38:17,390 [dynamic] arome_daily_2026-06-09.nc → 8 windows, 6.7 MB
2026-06-13 21:38:17,910 [dynamic] arome_daily_2026-06-10.nc → 8 windows, 6.2 MB
2026-06-13 21:38:18,408 [dynamic] arome_daily_2026-06-11.nc → 8 windows, 5.4 MB
2026-06-13 21:38:18,886 [dynamic] arome_daily_2026-06-12.nc → 8 windows, 5.1 MB
2026-06-13 21:38:19,268 [dynamic] arome_daily_2026-06-13.nc → 6 windows, 4.3 MB


---
## Cache loader for the pipeline

In [ ]:
def load_arome_daily_cache(
    start_date: date,
    end_date: date,
) -> xr.Dataset:
    """Load cached AROME daily grids. Prefers static, falls back to dynamic."""
    files = []
    missing = []
    current = start_date

    while current <= end_date:
        fname = f"arome_daily_{current.isoformat()}.nc"
        static = DAILY_DIR / "static" / fname
        dynamic = DAILY_DIR / "dynamic" / fname

        if static.exists():
            files.append(static)
        elif dynamic.exists():
            files.append(dynamic)
        else:
            missing.append(current)
        current += timedelta(days=1)

    if missing:
        logger.warning("%d missing days: %s%s", len(missing),
                        missing[:5], "..." if len(missing) > 5 else "")
    if not files:
        raise FileNotFoundError(f"No AROME for {start_date} → {end_date}")

    return xr.open_mfdataset(files, combine="by_coords")

In [ ]:
# Cache status
for mode in ["static", "dynamic"]:
    cached = sorted((DAILY_DIR / mode).glob("arome_daily_*.nc"))
    print(f"\n{mode.upper()}: {len(cached)} files")
    if cached:
        dates = [f.stem.replace('arome_daily_', '') for f in cached]
        total_mb = sum(f.stat().st_size for f in cached) / 1e6
        print(f"  {dates[0]} → {dates[-1]} ({total_mb:.0f} MB, {total_mb/max(len(cached),1):.1f} MB/day)")